# 06. Model Evaluation, Benchmarking & Champion Selection

This notebook queries trained model runs from Databricks MLflow tracking, computes serving complexity metrics (latency, memory size, throughput), performs business ROI simulations, and selects the production Champion model.

### Architectural Note on Environment Serialization
Models were trained in the local Python environment. To prevent cross-environment unpickling failures caused by differing Python and serialization runtime versions (e.g., Python 3.13 vs Python 3.12 serverless runtimes), model complexity benchmarks are executed in the training runtime. The finalized evaluation leaderboard is then published directly to Databricks Unity Catalog (`revenue_operations.gold.model_evaluation_leaderboard`) using the Databricks SDK.

## Phase 1: Retrieve MLflow Metrics & Run Metadata

In [0]:
# Phase 1: Connect to MLflow, Extract Runs, and Clean Leaderboard Metrics
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# 1. Connect to Databricks MLflow Tracking Server
mlflow.set_tracking_uri("databricks")
experiment_name = "/Users/prajwalparajuli2017@gmail.com/revenue_operations/delivery_risk"
experiment = mlflow.get_experiment_by_name(experiment_name)

# 2. Query all finished experiment runs
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.val_pr_auc DESC"]
)

# 3. Select and standardize metric and metadata columns
selected_columns = {
    'tags.mlflow.runName': 'model_name',
    'params.model_type': 'model_type',
    'metrics.val_pr_auc': 'val_pr_auc',
    'metrics.val_roc_auc': 'val_roc_auc',
    'metrics.val_f1_late': 'val_f1',
    'metrics.val_recall_late': 'val_recall',
    'metrics.val_precision_late': 'val_precision',
    'metrics.val_top10pct_capture': 'val_top10_capture',
    'metrics.train_f1_late': 'train_f1',
    'metrics.test_pr_auc': 'test_pr_auc',
    'metrics.test_f1_late': 'test_f1',
    'start_time': 'created_at',
    'run_id': 'run_id'
}

eval_df = runs_df[[col for col in selected_columns.keys() if col in runs_df.columns]].copy()
eval_df.rename(columns=selected_columns, inplace=True)

# 4. Compute Overfitting / Generalization Gap (Train F1 - Val F1)
eval_df['overfitting_gap'] = (eval_df['train_f1'] - eval_df['val_f1']).round(4)
eval_df = eval_df.sort_values(by='val_pr_auc', ascending=False).reset_index(drop=True)

# Display initial metrics table
eval_df

## Phase 2: Complexity & Serving Benchmarks (Latency, Memory, Throughput)

In [0]:
# Phase 2.1: Load Validation Sample Payload for Serving Benchmarks
import time
import sys
import pickle
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# Locate validation dataset
local_path_rel = Path("../data/external/val") if Path("../data/external/val").exists() else Path("data/external/val")
parquets = list(local_path_rel.glob("*.parquet"))
if not parquets:
    raise FileNotFoundError(f"No parquet files found in {local_path_rel}")

def sparse_to_dense(indices_col, values_col, size_col, prefix):
    size = int(size_col.iloc[0])
    n = len(indices_col)
    dense_matrix = np.zeros((n, size), dtype=np.float32)
    for row_idx, (idx_arr, val_arr) in enumerate(zip(indices_col, values_col)):
        if idx_arr is not None and len(idx_arr) > 0:
            for i, idx in enumerate(idx_arr):
                if idx is not None:
                    val = 1.0
                    if val_arr is not None and i < len(val_arr) and val_arr[i] is not None:
                        val = float(val_arr[i])
                    dense_matrix[row_idx, int(idx)] = val
    return pd.DataFrame(dense_matrix, columns=[f"{prefix}_{i}" for i in range(size)])

raw_val = pd.concat([pd.read_parquet(p, engine="fastparquet") for p in parquets], ignore_index=True)

extra_dfs = []
vector_cols_to_drop = []
for prefix in ["customer_state_encoded", "primary_product_category_encoded"]:
    if f"{prefix}.indices" in raw_val.columns:
        clean_name = "customer_state" if "customer_state" in prefix else "product_category"
        expanded = sparse_to_dense(raw_val[f"{prefix}.indices"], raw_val[f"{prefix}.values"], raw_val[f"{prefix}.size"], clean_name)
        extra_dfs.append(expanded)
        vector_cols_to_drop.extend([f"{prefix}.type", f"{prefix}.size", f"{prefix}.indices", f"{prefix}.values"])

exclude = ["order_id", "late_delivery_flag_indexed"] + vector_cols_to_drop
feature_cols = [c for c in raw_val.columns if c not in exclude]
X_val_sample = raw_val[feature_cols].copy()
if extra_dfs:
    X_val_sample = pd.concat([X_val_sample] + extra_dfs, axis=1)

# Prepare single payload (API simulation) and batch payload (1,000 records)
single_record = X_val_sample.iloc[[0]]
batch_1k = X_val_sample.head(1000)

# Scaler for models requiring normalized inputs
scaler = StandardScaler()
scaler.fit(X_val_sample)
single_record_scaled = scaler.transform(single_record)
batch_1k_scaled = scaler.transform(batch_1k)

print(f"Validation payload ready: {X_val_sample.shape[1]} features")


In [0]:
# Phase 2.2: Run Latency, Model Size & Throughput Benchmarks
benchmark_records = []

for idx, row in eval_df.iterrows():
    run_id = row['run_id']
    model_name = row['model_name']
    
    raw_run = runs_df[runs_df['run_id'] == run_id].iloc[0]
    train_duration_sec = None
    if pd.notnull(raw_run.get('start_time')) and pd.notnull(raw_run.get('end_time')):
        train_duration_sec = round((raw_run['end_time'] - raw_run['start_time']).total_seconds(), 1)
        
    print(f"Benchmarking: {model_name}...")
    model_size_mb, p50_latency, p95_latency, throughput = None, None, None, None
    
    try:
        if "ensemble" in model_name and "test" not in model_name:
            # Load ensemble sub-models
            lr_sub = mlflow.sklearn.load_model(f"runs:/{run_id}/logistic_regression_model")
            xgb_sub = mlflow.xgboost.load_model(f"runs:/{run_id}/xgboost_model")
            
            model_size_mb = round((sys.getsizeof(pickle.dumps(lr_sub)) + sys.getsizeof(pickle.dumps(xgb_sub))) / (1024 * 1024), 2)
            
            def predict_ensemble(X_raw, X_sc):
                p_lr = lr_sub.predict_proba(X_sc)[:, 1]
                p_xgb = xgb_sub.predict_proba(X_raw)[:, 1]
                return 0.5 * p_lr + 0.5 * p_xgb
            
            # Warmup
            for _ in range(10):
                _ = predict_ensemble(single_record, single_record_scaled)
                
            # Single-record latency (50 iterations)
            latencies = []
            for _ in range(50):
                t0 = time.perf_counter_ns()
                _ = predict_ensemble(single_record, single_record_scaled)
                t1 = time.perf_counter_ns()
                latencies.append((t1 - t0) / 1e6)
                
            p50_latency = round(np.percentile(latencies, 50), 2)
            p95_latency = round(np.percentile(latencies, 95), 2)
            
            # Batch throughput
            t0 = time.perf_counter()
            _ = predict_ensemble(batch_1k, batch_1k_scaled)
            t1 = time.perf_counter()
            throughput = round(len(batch_1k) / (t1 - t0), 0)
            
        elif "test_evaluation" in model_name:
            # Evaluation-only run
            model_size_mb, p50_latency, p95_latency, throughput = None, None, None, None
            
        else:
            # Single model
            model = mlflow.pyfunc.load_model(f"runs:/{run_id}/model")
            model_size_mb = round(sys.getsizeof(pickle.dumps(model._model_impl)) / (1024 * 1024), 2)
            
            for _ in range(10):
                _ = model.predict(single_record)
                
            latencies = []
            for _ in range(50):
                t0 = time.perf_counter_ns()
                _ = model.predict(single_record)
                t1 = time.perf_counter_ns()
                latencies.append((t1 - t0) / 1e6)
                
            p50_latency = round(np.percentile(latencies, 50), 2)
            p95_latency = round(np.percentile(latencies, 95), 2)
            
            t0 = time.perf_counter()
            _ = model.predict(batch_1k)
            t1 = time.perf_counter()
            throughput = round(len(batch_1k) / (t1 - t0), 0)
            
    except Exception as e:
        pass
        
    benchmark_records.append({
        'run_id': run_id,
        'train_duration_sec': train_duration_sec,
        'model_size_mb': model_size_mb,
        'latency_p50_ms': p50_latency,
        'latency_p95_ms': p95_latency,
        'batch_throughput_rps': throughput
    })

# Merge benchmark results into leaderboard
benchmark_df = pd.DataFrame(benchmark_records)
leaderboard_df = eval_df.merge(benchmark_df, on='run_id')

leaderboard_df[['model_name', 'val_pr_auc', 'val_top10_capture', 'overfitting_gap', 'model_size_mb', 'latency_p95_ms', 'batch_throughput_rps', 'train_duration_sec']]

## Phase 3: Business ROI Calculation & Multi-Criteria Champion Selection

In [0]:
# Phase 3.1: Financial Payoff Simulation & Composite Production Scoring
# 1. Business Cost Parameters (Per 10,000 Orders Baseline)
BASELINE_ORDERS = 10000
LATE_RATE = 0.08  # Baseline late delivery rate (~8%)
TOTAL_LATE_ORDERS = BASELINE_ORDERS * LATE_RATE # 800 late orders

COST_MISSED_LATE = 45.00   # Financial cost per late delivery (ticket handling, refunds, churn)
COST_INTERVENTION = 4.00   # Cost per proactive alert and logistics expedited reroute

# 2. Financial Payoff Simulation (Top 10% Operations Capacity = 1,000 alerts)
leaderboard_df['orders_inspected'] = BASELINE_ORDERS * 0.10 
leaderboard_df['late_orders_caught'] = (TOTAL_LATE_ORDERS * leaderboard_df['val_top10_capture']).round(0)
leaderboard_df['gross_loss_prevented'] = leaderboard_df['late_orders_caught'] * COST_MISSED_LATE
leaderboard_df['intervention_cost'] = leaderboard_df['orders_inspected'] * COST_INTERVENTION
leaderboard_df['net_business_value'] = (leaderboard_df['gross_loss_prevented'] - leaderboard_df['intervention_cost']).round(2)

# 3. Composite Production Score (0 to 100 Scale)
# Weights: PR-AUC (40%), Top-10% Capture (30%), Generalization Stability (15%), Latency SLA (15%)
max_pr = leaderboard_df['val_pr_auc'].max() if leaderboard_df['val_pr_auc'].max() > 0 else 1
max_cap = leaderboard_df['val_top10_capture'].max() if leaderboard_df['val_top10_capture'].max() > 0 else 1

leaderboard_df['composite_score'] = (
    0.40 * (leaderboard_df['val_pr_auc'] / max_pr) +
    0.30 * (leaderboard_df['val_top10_capture'] / max_cap) +
    0.15 * (1 - leaderboard_df['overfitting_gap'].clip(0, 1)) +
    0.15 * (1 - (leaderboard_df['latency_p95_ms'].fillna(15).clip(0, 50) / 50))
) * 100
leaderboard_df['composite_score'] = leaderboard_df['composite_score'].round(2)

# 4. Production Deployment Status Assignment
def assign_status(row, top_score):
    if pd.isnull(row['val_pr_auc']):
        return "Test Evaluation Split"
    elif row['composite_score'] == top_score:
        return "CHAMPION (Deploy)"
    elif row['composite_score'] >= top_score * 0.88:
        return "CHALLENGER (Standby)"
    else:
        return "REJECTED"

valid_scores = leaderboard_df.dropna(subset=['val_pr_auc'])['composite_score']
top_score = valid_scores.max() if len(valid_scores) > 0 else 0
leaderboard_df['deployment_status'] = leaderboard_df.apply(lambda r: assign_status(r, top_score), axis=1)

# Display Decision Matrix
display_cols = [
    'model_name', 
    'val_pr_auc', 
    'val_top10_capture', 
    'overfitting_gap',
    'net_business_value', 
    'latency_p95_ms', 
    'composite_score', 
    'deployment_status'
]
leaderboard_df[display_cols].sort_values(by='composite_score', ascending=False).reset_index(drop=True)

In [0]:
# Phase 3.2: Pareto Frontier & Financial Savings Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 6))

# 1. Pareto Frontier: PR-AUC vs. p95 Latency
plt.subplot(1, 2, 1)
plot_data = leaderboard_df.dropna(subset=["val_pr_auc", "latency_p95_ms"])
palette_map = {"CHAMPION (Deploy)": "#2ecc71", "CHALLENGER (Standby)": "#3498db", "REJECTED": "#e74c3c"}
sns.scatterplot(
    data=plot_data,
    x="latency_p95_ms",
    y="val_pr_auc",
    hue="deployment_status",
    style="deployment_status",
    s=220,
    palette=palette_map
)

for _, r in plot_data.iterrows():
    short_name = r["model_name"].replace("_with_tuning", "").replace("model_", "").replace("logistic_regression_", "logreg_")
    plt.text(r["latency_p95_ms"] + 0.3, r["val_pr_auc"], short_name, fontsize=9)

plt.axvline(x=15, color="gray", linestyle="--", label="15ms SLA Target")
plt.title("Pareto Frontier: Performance vs. Serving Latency", fontsize=12, fontweight="bold")
plt.xlabel("p95 Latency (ms) [Lower is Better]")
plt.ylabel("Val PR-AUC [Higher is Better]")
plt.grid(True, alpha=0.3)
plt.legend(loc="lower right")

# 2. Net Business Savings Bar Chart
plt.subplot(1, 2, 2)
valid_biz = leaderboard_df.dropna(subset=["net_business_value"]).sort_values("net_business_value", ascending=False)
sns.barplot(
    data=valid_biz,
    x="net_business_value",
    y="model_name",
    hue="model_name",
    palette="Blues_r",
    legend=False
)
plt.title("Net Financial Savings ($) per 10,000 Orders", fontsize=12, fontweight="bold")
plt.xlabel("Net Savings ($) [Higher is Better]")
plt.ylabel("")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Phase 4: Production Champion Deployment Card & Unity Catalog Delta Lake Export

### Publishing Strategy
To ensure consistent artifact consumption across downstream Databricks services (such as Lakeview Dashboards and Model Serving pipelines), the benchmark evaluation leaderboard is published directly to Databricks Unity Catalog (`revenue_operations.gold.model_evaluation_leaderboard`).

The code below uses the Databricks SDK to:
1. Serialize the in-memory benchmark DataFrame to Parquet.
2. Upload the file to the Unity Catalog Volume: `/Volumes/revenue_operations/gold/ml_datasets/model_evaluation_leaderboard.parquet`.
3. Execute a `CREATE OR REPLACE TABLE` statement on the Databricks SQL Warehouse to register the Delta table.

In [0]:
# Phase 4.1: Production Deployment Specification (Champion Card)
champion_row = leaderboard_df[leaderboard_df['deployment_status'] == 'CHAMPION (Deploy)'].iloc[0]

print("=" * 65)
print(f"PRODUCTION DEPLOYMENT CARD: {champion_row['model_name'].upper()}")
print("=" * 65)
print(f"Deployment Role:         Production Champion (@champion alias)")
print(f"Model Architecture:      {champion_row['model_type']}")
print(f"MLflow Run ID:           {champion_row['run_id']}")
print(f"Primary Metric (PR-AUC): {champion_row['val_pr_auc']:.4f}")
print(f"Top-10% Capture Rate:    {champion_row['val_top10_capture']:.1%} of late shipments caught")
print(f"Serving p95 Latency:     {champion_row['latency_p95_ms']} ms (SLA Budget: < 15 ms)")
print(f"Model Storage Size:      {champion_row['model_size_mb']} MB")
print(f"Net Value Generated:     +${champion_row['net_business_value']:,.2f} per 10,000 orders")
print("=" * 65)

In [0]:
# Phase 4.2: Push Benchmark Leaderboard to Databricks Unity Catalog Delta Table
import io
from databricks.sdk import WorkspaceClient

# 1. Initialize Databricks Workspace Client
w = WorkspaceClient()

# 2. Serialize leaderboard DataFrame to in-memory Parquet
parquet_buffer = io.BytesIO()
leaderboard_df.to_parquet(parquet_buffer, engine='pyarrow', index=False)
parquet_buffer.seek(0)

# 3. Upload to Unity Catalog Volume
target_volume_file = "/Volumes/revenue_operations/gold/ml_datasets/model_evaluation_leaderboard.parquet"
w.files.upload(target_volume_file, parquet_buffer, overwrite=True)
print(f"Uploaded parquet to Databricks Volume: {target_volume_file}")

# 4. Execute SQL statement on Databricks SQL Warehouse to create the Delta Table
warehouse = list(w.warehouses.list())[0]
print(f"Executing SQL on warehouse: {warehouse.name}")

create_table_sql = f"""
CREATE OR REPLACE TABLE revenue_operations.gold.model_evaluation_leaderboard AS
SELECT * FROM read_files('{target_volume_file}', format => 'parquet');
"""

stmt = w.statement_execution.execute_statement(
    warehouse_id=warehouse.id,
    statement=create_table_sql,
    wait_timeout='50s'
)

print("Delta Table live in Unity Catalog: revenue_operations.gold.model_evaluation_leaderboard")